# Glioma: Segmentação de Sub-regiões + Graduação (SAM 2 / MedSAM + Atenção + LoRA)

**Projeto final — Visão Computacional Avançada · FCTE/UnB**

> Pipeline de duas etapas *acopladas* que **(A) segmenta** as sub-regiões do tumor
> (necrose/NCR, edema/ED, tumor ativo/ET) e **(B) gradua** a lesão (LGG × HGG) a
> partir da máscara segmentada. A graduação depende explicitamente da segmentação
> — **não** é um classificador de imagem isolado.

---

## Resumo (o que este notebook entrega)

| # | Entregável | Seção |
|---|------------|-------|
| 1 | Segmentação multi-classe com **Attention U-Net** sobre encoder de fundação (SAM 2/MedSAM) adaptado por **LoRA** | 3, 7 |
| 2 | **Graduação guiada pela máscara** (masked-attention pooling + descritores geométricos) | 7 |
| 3 | Métricas do desafio BraTS: **Dice / IoU / Sens / Spec / HD95** por sub-região (WT/TC/ET) | 8 |
| 4 | Graduação: **AUC/ROC**, matriz de confusão, F1, e **calibração (ECE, Brier, reliability)** | 8 |
| 5 | **Ablação científica**: encoder de fundação (LoRA) × congelado × U-Net do zero | 9 |
| 6 | **Biomarcadores tumorais** derivados da segmentação → ponte interpretável seg→grau | 10 |
| 7 | **Incerteza** (MC-Dropout + TTA): mapas de "onde o modelo hesita" | 11 |
| 8 | **Explicabilidade** da graduação (atenção intrínseca + saliência por oclusão) | 12 |

**Pronto para a GPU T4 do Colab.** O notebook roda ponta-a-ponta em dados
**sintéticos** (sem download) para validar tudo; troque uma flag para treinar no
**BraTS real**.

### Como usar
1. `Runtime → Change runtime type → T4 GPU`.
2. Rode as células **1–3** (setup + sanity-check sintético).
3. Para dados reais: rode a **Seção 4** (download BraTS2020 via Kaggle) e ajuste
   `DATA_SOURCE = "brats"` na Seção 5.
4. Rode 5→12 para treino, avaliação e todas as análises.

## 1. Ambiente, GPU e dependências

In [ ]:
# --- No Colab: clona o repositório (idempotente -- pula se já clonado) ---
# IMPORTANTE: "!comando && %magic" NAO funciona -- '%cd' e' magic do IPython,
# nao comando de shell; misturado com '!' e '&&' o bash tenta executa-lo e
# falha (erro "fg: no job control") sem trocar de pasta de verdade. Por isso
# o clone e o %cd vao em linhas separadas abaixo.
REPO_ROOT = '/content/glioma_seg_grade'   # ajuste se clonar em outro lugar
![ -d glioma_seg_grade ] || git clone https://github.com/Bappoz/glioma-seg-grad glioma_seg_grade
%cd glioma_seg_grade
!pip -q install -r requirements.txt

import os, sys
# path ABSOLUTO (nao '.') -- continua valido mesmo que o cwd mude depois, e
# permite que outras celulas (ex.: download do Kaggle) re-afirmem o path sem
# depender desta celula ter rodado antes na MESMA sessao do kernel.
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch, platform
print('Python :', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Reprodutibilidade
import numpy as np, random
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)

# matplotlib inline + estilo limpo
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})

## 2. Metodologia

### Arquitetura em dois estágios acoplados

**Estágio A — Segmentação.** Um encoder de fundação (SAM 2 / MedSAM 2, backbone
*Hiera*) — **congelado** — extrai features multi-escala. Um decoder com
**Attention Gates** (Oktay et al., 2018) funde essas escalas por *skip-connections*
e reconstrói a máscara multi-classe {fundo, NCR/NET, ED, ET}.

**Estágio B — Graduação.** As features do encoder são agregadas por um
**masked-attention pooling** guiado pela máscara predita no Estágio A e
concatenadas a **descritores geométricos** (frações de área por sub-região). A
saída é o grau (LGG × HGG). A decisão só "olha" para o tumor segmentado → decisão
interpretável e alinhada à clínica.

### Por que SAM 2 + atenção supera uma U-Net pura?
- O encoder Hiera vem **pré-treinado em >1 bilhão de máscaras**: traz priors de
  bordas/objetidade que uma U-Net do zero em poucas centenas de volumes não aprende.
- **Attention Gates** suprimem crânio/edema difuso → bordas mais nítidas.
- O **acoplamento seg→grau** torna a graduação dependente da região segmentada.

### Fine-tuning eficiente com LoRA
O encoder fica congelado; injetamos adaptadores de baixo posto (LoRA) nas
projeções de atenção. Treinam ~1–5% dos pesos → **cabe na T4** e **preserva os
priors** (evita *catastrophic forgetting*). `lora_B=0` no início ⇒ o modelo parte
exatamente do encoder pré-treinado; `merge_all_lora()` funde para inferência sem custo.

### Perdas e métricas
- Segmentação: **Dice + CE** (padrão-ouro) ou **Focal-Tversky** (β>α prioriza
  *recall* no ET pequeno).
- Graduação: **CE com label smoothing** (melhora a calibração).
- Métricas: Dice/IoU/Sens/Spec/HD95 (seg) · Acc/AUC/F1/ECE/Brier (grau).

In [ ]:
# Diagrama da arquitetura (renderizado com matplotlib — sem dependências externas)
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def draw_architecture():
    fig, ax = plt.subplots(figsize=(12, 4.2)); ax.axis('off')
    ax.set_xlim(0, 12); ax.set_ylim(0, 4.2)
    def box(x, y, w, h, text, color):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.03',
                     linewidth=1.4, edgecolor='#333', facecolor=color, alpha=0.9))
        ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=9.5)
    def arrow(x1, y1, x2, y2):
        ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-|>',
                     mutation_scale=14, linewidth=1.4, color='#444'))
    box(0.2, 1.6, 1.9, 1.0, 'MRI\n(FLAIR,T1ce,T2)', '#DCE6F5')
    box(2.6, 1.4, 2.3, 1.4, 'Encoder Hiera\n(SAM2/MedSAM)\n❄ congelado + LoRA', '#F5E6CC')
    box(5.6, 2.4, 2.3, 1.1, 'Attention U-Net\n(decoder + gates)', '#D8EAD3')
    box(5.6, 0.3, 2.3, 1.1, 'Masked-Attention\nPooling + geometria', '#F3D9DE')
    box(8.7, 2.4, 3.0, 1.1, 'Máscara: NCR/ED/ET\n(WT · TC · ET)', '#D8EAD3')
    box(8.7, 0.3, 3.0, 1.1, 'Grau: LGG × HGG\n(+ incerteza)', '#F3D9DE')
    arrow(2.1, 2.1, 2.6, 2.1)
    arrow(4.9, 2.2, 5.6, 2.8); arrow(4.9, 1.9, 5.6, 0.9)
    arrow(7.9, 2.9, 8.7, 2.9); arrow(7.9, 0.85, 8.7, 0.85)
    arrow(7.0, 2.4, 6.9, 1.4)  # máscara guia a graduação (acoplamento)
    ax.text(6.55, 1.9, 'acopla', fontsize=8, color='#C0392B', rotation=90, va='center')
    ax.set_title('Pipeline: segmentação (topo) → graduação guiada pela máscara (base)', fontsize=11)
    plt.tight_layout(); return fig

draw_architecture(); plt.show()

## 3. Sanity-check com dados sintéticos (valida o pipeline sem download)

`SyntheticBraTS` gera anéis aninhados (edema → core → ET) como "tumor"; ~40% dos
casos são LGG-like (sem ET), dividindo o rótulo de grau em 2 classes para a
curva ROC funcionar. Serve para confirmar shapes e o loop antes do BraTS real.

In [ ]:
from src.train import Trainer, TrainConfig
from src.dataset import SyntheticBraTS, SegAugmentation
from torch.utils.data import DataLoader
from src import viz

sanity_cfg = TrainConfig(backbone='stub', epochs=3, batch_size=16, img_size=128,
                         region='tversky', p_drop=0.1, warmup_epochs=1,
                         out_dir='runs/sanity')
tr = DataLoader(SyntheticBraTS(96, img_size=128, transform=SegAugmentation()),
                batch_size=16, shuffle=True)
va = DataLoader(SyntheticBraTS(32, img_size=128), batch_size=16)
sanity_hist = Trainer(sanity_cfg).fit(tr, va)
viz.plot_training_curves(sanity_hist); plt.show()
print('Pipeline OK — shapes e loop validados.')

## 4. Aquisição do BraTS real

> Pule esta seção se quiser apenas a demonstração sintética. Para treinar de
> verdade, escolha **uma** das opções abaixo. A **Opção A (Kaggle BraTS2020)** é
> a mais simples e é a que este notebook cabeia diretamente.

### Opção A — Kaggle · BraTS2020 (recomendada)
Mirror estável (`awsaf49/brats20-dataset-training-validation`, 369 casos) com a
estrutura oficial `BraTS20_Training_XXX/*_flair.nii …` **e** o grau real em
`name_mapping.csv` (coluna `Grade` = HGG/LGG). Requer um token do Kaggle:
**Kaggle → Account → Create New API Token** → baixa `kaggle.json` → suba para o Colab.

> **Procedência:** mirror de terceiros (não o portal oficial do desafio) — declare
> isso no relatório.

In [ ]:
# --- Opção A: download do BraTS2020 via API do Kaggle ---
# Guarda defensiva: se esta célula for executada isoladamente (ex.: depois de
# reiniciar o runtime e pular direto para cá, sem re-rodar a Seção 1), garante
# que 'src' continue importável mesmo sem ter rodado a célula de setup antes.
import os, sys
_repo_root = '/content/glioma_seg_grade'
if os.path.isdir(_repo_root) and _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

# 1) autenticação SEM subir arquivo: usa o Colab Secrets (recomendado) com
#    fallback para digitar na hora (getpass -> nunca aparece na tela nem fica
#    salvo no notebook). Configure uma vez em Secrets (ícone 🔑 na barra
#    lateral esquerda): KAGGLE_USERNAME e KAGGLE_KEY.
import json as _json, getpass

try:
    from google.colab import userdata
    kaggle_user = userdata.get('KAGGLE_USERNAME')
    kaggle_key = userdata.get('KAGGLE_KEY')
except Exception:
    kaggle_user = kaggle_key = None

if not kaggle_user or not kaggle_key:
    print("Dica: salve KAGGLE_USERNAME/KAGGLE_KEY no painel Secrets (🔑) para "
          "não digitar isso toda sessão.")
    kaggle_user = input('Usuário Kaggle: ')
    kaggle_key = getpass.getpass('Token/API key do Kaggle: ')

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    _json.dump({"username": kaggle_user, "key": kaggle_key}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
del kaggle_key   # não mantém o token em memória além do necessário

# 2) download
!pip -q install kaggle
!kaggle datasets download -d awsaf49/brats20-dataset-training-validation -p /content --unzip

# 3) auto-descoberta da raiz (robusto a variações de path do unzip do Kaggle)
from src.dataset import find_brats_root, build_grade_lookup_from_csv, discover_cases
BRATS_ROOT = find_brats_root('/content')
print('BRATS_ROOT:', BRATS_ROOT, '|', len(discover_cases(BRATS_ROOT)), 'casos encontrados')

# 4) grau real do name_mapping.csv; se a coluna não existir nesse mirror,
#    cai automaticamente no proxy fraco (presença de ET) sem quebrar o pipeline.
try:
    grade_lookup = build_grade_lookup_from_csv(f'{BRATS_ROOT}/name_mapping.csv', root=BRATS_ROOT)
    print(f'grau REAL lido de name_mapping.csv: {len(grade_lookup)} casos com grau')
except (FileNotFoundError, KeyError) as e:
    print(f'name_mapping.csv indisponível/sem coluna esperada ({e!r}) '
          f'-> usando proxy fraco (presença de ET) a nível de paciente.')
    grade_lookup = None
import collections
print('balanço (0=LGG,1=HGG):', dict(collections.Counter((grade_lookup or {}).values())))

### Opção B — Kaggle · BraTS2018 (pastas HGG/LGG)
Mirrors do BraTS2018 preservam as pastas `HGG/` e `LGG/` (grau pela pasta) com
nomenclatura canônica. Confirme o `owner/dataset` no Kaggle e a estrutura.

In [ ]:
# --- Opção B: BraTS2018 organizado em HGG/ e LGG/ ---
# !kaggle datasets download -d <owner>/<brats2018> -p /content --unzip
# from src.dataset import build_grade_dataset_from_folders
# grade_lookup = build_grade_dataset_from_folders(
#     [('/content/MICCAI_BraTS_2018_Data_Training/HGG', 1),
#      ('/content/MICCAI_BraTS_2018_Data_Training/LGG', 0)],
#     out_root='/content/BraTS_merged')
# BRATS_ROOT = '/content/BraTS_merged'

### Opção C — TCIA (BRATS-TCGA-LGG/GBM)
Nomenclatura diferente (`t1Gd`, `GlistrBoost`) e download via **Aspera Connect**.
Use `build_tcia_grade_dataset(lgg_dir, gbm_dir, out_root)` (normaliza os nomes por
symlink). Recomendado só se A/B falharem.

## 5. Configuração do experimento e loaders

Uma única flag controla a fonte de dados. Com `DATA_SOURCE = "synthetic"` o
notebook roda inteiro sem download (útil para revisar tudo). Com `"brats"`, ele
usa o **caminho pré-computado** (`precompute_slices`): extrai as fatias
informativas **uma vez** para shards `.npz` compactos → treino muito mais rápido
na T4 (sem decodificar NIfTI a cada passo) e RAM previsível.

In [ ]:
DATA_SOURCE = 'brats'       # 'synthetic'  |  'brats'
BACKBONE    = 'stub'        # 'stub' (sem checkpoint)  |  'sam2' (com checkpoint Hiera)

from src.train import TrainConfig
cfg = TrainConfig(
    backbone=BACKBONE,
    img_size=128 if DATA_SOURCE == 'synthetic' else 256,
    batch_size=16 if BACKBONE == 'stub' else 4,   # T4: ~4 com SAM2+LoRA
    epochs=10,
    region='tversky', tversky_alpha=0.3, tversky_beta=0.7,
    use_lora=True, lora_r=4, p_drop=0.1,
    warmup_epochs=2, grad_clip=1.0, accum_steps=2,
    seg_ce_weight=[0.1, 1.0, 1.0, 1.0],
    early_stop_patience=6,
    out_dir='runs/exp1',
)
cfg.device = DEVICE
print(cfg)

In [ ]:
from torch.utils.data import DataLoader
from src.dataset import (SyntheticBraTS, SegAugmentation,
                         precompute_slices, make_loaders_precomputed)

if DATA_SOURCE == 'synthetic':
    train_dl = DataLoader(SyntheticBraTS(320, img_size=cfg.img_size, transform=SegAugmentation()),
                          batch_size=cfg.batch_size, shuffle=True)
    val_dl   = DataLoader(SyntheticBraTS(96, img_size=cfg.img_size),
                          batch_size=cfg.batch_size)
else:
    # 1) pré-computa as fatias uma vez -> shards .npz (reexecutar é no-op se já existir)
    SHARD_DIR = '/content/brats_shards'
    precompute_slices(BRATS_ROOT, SHARD_DIR, slices_per_case=cfg.slices_per_case,
                      img_size=cfg.img_size, grade_lookup=grade_lookup)
    # 2) loaders com split por paciente estratificado + augmentation + balanceamento
    train_dl, val_dl = make_loaders_precomputed(SHARD_DIR, batch_size=cfg.batch_size,
                                                num_workers=2, augment=True,
                                                balance_grade=True)

print('batches treino/val:', len(train_dl), '/', len(val_dl))

## 5.1 Exploração do dataset (EDA)

Estatísticas básicas e exemplos visuais — orienta escolhas de pré-processamento e
evidencia o desbalanço fundo/tumor típico do BraTS.

In [ ]:
import numpy as np, collections

# balanço de grau e área tumoral média por classe (amostra do loader de validação)
grades, areas = [], collections.Counter()
n_px = 0
for b in val_dl:
    grades += b['grade'].tolist()
    for c in range(4):
        areas[c] += int((b['seg_mask'] == c).sum())
    n_px += b['seg_mask'].numel()
    if len(grades) >= 200: break

print('Balanço de grau (0=LGG, 1=HGG):', dict(collections.Counter(grades)))
names = {0: 'fundo', 1: 'NCR/NET', 2: 'ED', 3: 'ET'}
print('Fração de pixels por classe:')
for c in range(4):
    print(f'  {names[c]:8s}: {100*areas[c]/max(n_px,1):5.2f}%')
print('→ o tumor ocupa uma fração pequena: justifica Dice/Tversky + peso no CE.')

In [ ]:
# Exemplos: MRI + máscara sobreposta (3 casos do loader de validação)
from src import viz
batch = next(iter(val_dl))
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i in range(3):
    viz.overlay_mask(axes[i], batch['image'][i, 0], batch['seg_mask'][i],
                     f"caso {i} · grau={'HGG' if int(batch['grade'][i]) else 'LGG'}")
plt.tight_layout(); plt.show()

## 6. Treino

Treina o modelo principal (encoder de fundação + LoRA + Attention U-Net +
graduação guiada). Salva `runs/exp1/checkpoints/best.pt` (+ `lora.pt` leve) e o
histórico. **Para usar o SAM 2 real:** instale o repo, baixe um checkpoint Hiera
e ajuste `cfg.backbone='sam2'`, `cfg.sam2_cfg`, `cfg.sam2_ckpt` (ver README).

In [ ]:
from src.train import Trainer
set_seed(42)
trainer = Trainer(cfg)
n_train = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
print(f'Parâmetros treináveis: {n_train/1e6:.2f}M  (encoder de fundação congelado)')
history = trainer.fit(train_dl, val_dl)
model = trainer.model

In [ ]:
viz.plot_training_curves(history, save_path='curvas_treino.png'); plt.show()

## 7. Resultados quantitativos

Acumula as predições da validação e reporta as métricas do desafio BraTS por
sub-região, além de AUC/ROC e **calibração** da graduação.

In [ ]:
import torch

@torch.no_grad()
def collect_predictions(model, loader, device, max_batches=None):
    model.eval()
    P, G, GL, GT, IMG, CID = [], [], [], [], [], []
    for i, b in enumerate(loader):
        out = model(b['image'].to(device), return_aux=False)
        P.append(out['seg_logits'].argmax(1).cpu()); G.append(b['seg_mask'])
        GL.append(out['grade_logits'].cpu()); GT.append(b['grade'])
        IMG.append(b['image']); CID += list(b['case_id'])
        if max_batches and i + 1 >= max_batches: break
    return (torch.cat(P), torch.cat(G), torch.cat(GL), torch.cat(GT),
            torch.cat(IMG), CID)

pred, gt, grade_logits, grade_true, images, case_ids = collect_predictions(model, val_dl, DEVICE)
print('amostras de validação avaliadas:', pred.shape[0])

In [ ]:
from src.metrics import seg_scores, seg_scores_hd95, grade_scores, grade_roc_auc
import numpy as np

seg = seg_scores(pred, gt)
print('===== SEGMENTAÇÃO (por sub-região) =====')
print(f"{'região':6s} {'Dice':>7s} {'IoU':>7s} {'Sens':>7s} {'Spec':>7s}")
for r in ('WT', 'TC', 'ET'):
    print(f"{r:6s} {seg[f'dice_{r}']:7.3f} {seg[f'iou_{r}']:7.3f} "
          f"{seg[f'sens_{r}']:7.3f} {seg[f'spec_{r}']:7.3f}")
print(f"{'MÉDIA':6s} {seg['dice_mean']:7.3f}")

# HD95 numa subamostra (custoso): média sobre casos com máscara não-vazia
hd = seg_scores_hd95(pred[:64], gt[:64])
print('\nHausdorff95 (px):', {k: round(v, 2) for k, v in hd.items()})
viz.plot_hd95_bars(hd); plt.show()

In [ ]:
# Graduação: acurácia, AUC, ROC, relatório e matriz de confusão
from src.metrics import grade_report
gs = grade_scores(grade_logits, grade_true)
roc = grade_roc_auc(grade_logits, grade_true)
rep = grade_report(grade_logits, grade_true)
print('===== GRADUAÇÃO (LGG × HGG) =====')
print(f"Accuracy       : {gs['grade_acc']:.3f}")
print(f"Sensibilidade  : {gs.get('grade_sens', float('nan')):.3f}")
print(f"Especificidade : {gs.get('grade_spec', float('nan')):.3f}")
print(f"AUC            : {roc['auc']:.3f}")
print(f"F1 (macro)     : {rep.get('f1_macro', float('nan')):.3f}")
print(f"Matriz de confusão [LGG,HGG]: {rep.get('confusion')}")
viz.plot_roc(roc); plt.show()

In [ ]:
# Calibração: a probabilidade de grau é honesta? (importante p/ apoio à decisão)
from src.metrics import expected_calibration_error, brier_score, reliability_bins
ece = expected_calibration_error(grade_logits, grade_true)
brier = brier_score(grade_logits, grade_true)
bins = reliability_bins(grade_logits, grade_true, n_bins=10)
print(f'ECE (Expected Calibration Error): {ece:.3f}   (0 = perfeito)')
print(f'Brier score                    : {brier:.3f}   (menor é melhor)')
viz.plot_reliability(bins, ece); plt.show()

## 8. Ablação científica

Isola o efeito do **encoder de fundação + LoRA** treinando três variantes com o
mesmo Trainer/loaders. A única coisa que muda é o extrator de features:

| Variante | Encoder |
|----------|---------|
| `unet` | U-Net treinada do zero (sem fundação) |
| `stub_frozen` | encoder de fundação **congelado**, sem adaptação |
| `stub_lora` | encoder de fundação adaptado por **LoRA** |

> **Nota:** no modo `stub` o encoder "congelado" tem pesos aleatórios (o stub não
> é pré-treinado), então `stub_frozen` fica fraco — ele existe para demonstrar o
> **ganho da adaptação LoRA**. Com o **SAM 2 real**, `frozen` já parte de features
> fortes; troque para a variante `sam2_lora` para a comparação definitiva.

In [ ]:
from src.baseline import run_ablation
from dataclasses import replace

abl_cfg = replace(cfg, epochs=6, out_dir='runs/ablacao')   # menos épocas p/ a T4
ablation = run_ablation(abl_cfg, train_dl, val_dl,
                        variants=['unet', 'stub_frozen', 'stub_lora'])

cols = ['variant', 'params_M', 'dice_mean', 'dice_WT', 'dice_TC', 'dice_ET',
        'grade_acc', 'grade_auc']
try:
    import pandas as pd
    df = pd.DataFrame(ablation)[cols]
    display(df.style.hide(axis='index').format(precision=3))
except Exception:
    print(' | '.join(f'{c:>10s}' for c in cols))
    for r in ablation:
        print(' | '.join(f'{r[c]:>10}' if isinstance(r[c], str) else f'{r[c]:>10.3f}' for c in cols))
viz.plot_ablation(ablation, save_path='ablacao.png'); plt.show()

## 9. Biomarcadores tumorais (ponte interpretável seg→grau)

Transformamos a **máscara predita** em medidas clínicas (volumes de WT/TC/ET,
razão ET/TC, fração de necrose) e medimos o poder de cada uma para separar o
grau. Em seguida, um **classificador logístico transparente** só sobre esses
biomarcadores é comparado à cabeça neural — evidência de que a decisão de grau é
sustentada por geometria tumoral plausível.

In [ ]:
from src.biomarkers import (aggregate_biomarkers, biomarker_grade_association,
                            interpretable_grade_classifier)
import numpy as np, collections

# agrega as fatias preditas por caso -> uma máscara empilhada [S,H,W] por paciente
by_case = collections.defaultdict(list)
for k in range(pred.shape[0]):
    by_case[case_ids[k]].append(pred[k].numpy())
case_masks = {cid: np.stack(v) for cid, v in by_case.items()}
grade_by_case = {cid: int(grade_true[[i for i, c in enumerate(case_ids) if c == cid][0]])
                 for cid in case_masks}

rows = aggregate_biomarkers(case_masks, grade_by_case)
assoc = biomarker_grade_association(rows)
print('Poder discriminativo (AUC univariada) dos biomarcadores vs grau:')
for a in assoc[:6]:
    print(f"  {a['feature']:14s} AUC={a['auc']:.3f}  {a.get('direction','')}")
viz.plot_biomarker_associations(assoc); plt.show()
viz.plot_biomarker_by_grade(rows, feature=assoc[0]['feature']); plt.show()

In [ ]:
# Classificador logístico interpretável (só biomarcadores) vs cabeça neural
try:
    res = interpretable_grade_classifier(rows, features=('vol_ET', 'ratio_ET_TC', 'vol_TC'))
    print('Classificador logístico (biomarcadores):')
    print(f"  AUC (validação cruzada): {res['cv_auc']:.3f}")
    print('  coeficientes (importância):', {k: round(v, 2) for k, v in res['coefficients'].items()})
    print(f"\nCabeça neural de graduação: AUC = {roc['auc']:.3f}")
    print('→ concordância entre a decisão neural e a geometria tumoral interpretável.')
except Exception as e:
    print('classificador interpretável precisa de casos suficientes com ambos os graus:', e)

## 10. Incerteza (MC-Dropout + TTA)

**Onde o modelo hesita?** Mantendo o dropout ativo na inferência e fazendo N
passagens estocásticas (MC-Dropout), estimamos a incerteza **epistêmica**
(informação mútua). A **TTA** (média sobre flips) costuma elevar o Dice.

In [ ]:
from src.uncertainty import mc_dropout_predict, tta_predict, uncertainty_summary

# escolhe uma amostra de validação com tumor visível
idx = int(max(range(min(64, pred.shape[0])), key=lambda k: int((gt[k] > 0).sum())))
x = images[idx:idx+1].to(DEVICE)

mc = mc_dropout_predict(model, x, n_samples=15)
print('Incerteza (médias):', {k: round(v, 4) for k, v in uncertainty_summary(mc).items()})
viz.plot_uncertainty_panel(images[idx, 0], mc['seg_pred'][0].cpu(),
                           mc['seg_total_entropy'][0].cpu(),
                           mc['seg_mutual_info'][0].cpu(),
                           save_path='incerteza.png'); plt.show()

In [ ]:
# TTA: ganho de Dice ao promediar predições sobre flips
from src.metrics import seg_scores
base = seg_scores(pred[:64], gt[:64])['dice_mean']
tta_preds = []
for k in range(min(64, pred.shape[0])):
    tp = tta_predict(model, images[k:k+1].to(DEVICE))['seg_pred'][0].cpu()
    tta_preds.append(tp)
tta_dice = seg_scores(torch.stack(tta_preds), gt[:len(tta_preds)])['dice_mean']
print(f'Dice médio  —  base: {base:.3f}   |   com TTA: {tta_dice:.3f}   '
      f'(Δ = {tta_dice - base:+.3f})')

## 11. Explicabilidade da graduação

Qual região sustentou a decisão de grau? Sobrepomos (i) a **atenção intrínseca**
do masked-attention pooling (fiel ao que o modelo agregou) e (ii) a **saliência
por oclusão** (model-agnostic, valida a atenção).

In [ ]:
from src.explain import grade_attention, occlusion_saliency
import torch

attn = grade_attention(model, x)[0].cpu()
occ = occlusion_saliency(model, x, patch=max(16, cfg.img_size // 8),
                         stride=max(8, cfg.img_size // 16))[0].cpu()
gprob = torch.softmax(model(x, return_aux=False)['grade_logits'], 1)[0]
viz.plot_explanation_panel(images[idx, 0], attn, occ,
                           grade_pred=int(gprob.argmax()),
                           grade_prob=float(gprob.max()),
                           save_path='explicabilidade.png'); plt.show()

## 12. Visualização qualitativa da segmentação

Painel MRI | Ground Truth | Predição | **Mapa de erro** (FP magenta / FN ciano) —
evidência direta dos pontos fortes e fracos do modelo.

In [ ]:
from src.metrics import seg_scores
d = seg_scores(pred[idx:idx+1], gt[idx:idx+1])
gp = int(torch.softmax(grade_logits[idx:idx+1], 1).argmax())
viz.qualitative_panel(images[idx, 0], gt[idx], pred[idx],
                      grade_true=int(grade_true[idx]), grade_pred=gp,
                      dice_txt=f"Dice WT={d['dice_WT']:.2f} · TC={d['dice_TC']:.2f} · ET={d['dice_ET']:.2f}",
                      save_path='qualitativa.png'); plt.show()

## 13. Conclusão, limitações e trabalhos futuros

**O que foi demonstrado.** Um pipeline de duas etapas *acopladas* em que a
graduação depende explicitamente da segmentação, com adaptação eficiente por LoRA
(cabe na T4, treina ~1–5% dos pesos). Reportamos as métricas do desafio BraTS
(Dice/IoU/Sens/Spec/HD95), AUC/ROC e **calibração** da graduação, uma **ablação**
que isola o ganho do encoder de fundação, **biomarcadores** interpretáveis como
ponte seg→grau, **mapas de incerteza** (MC-Dropout/TTA) e **explicabilidade** da
decisão de grau.

**Limitações.**
- Segmentação **2D por fatia** (sem contexto 3D inter-fatias do MedSAM2 pleno).
- Se o subset não trouxer o grau OMS, o rótulo de grau é **proxy** (presença de ET)
  — declare no relatório. Com BraTS2020 (Opção A) o grau é **real** (name_mapping.csv).
- Validação em **um split**; para publicação, use validação cruzada por paciente.
- No modo `stub`, o encoder "congelado" é aleatório — a comparação definitiva de
  fundação exige o **SAM 2 real**.

**Trabalhos futuros.**
- Ligar o **SAM 2/MedSAM 2 real** (checkpoint Hiera) e a memória inter-fatias.
- **2.5D** (fatias vizinhas como canais) ou 3D leve para coerência volumétrica.
- **Uncertainty weighting** multi-tarefa (Kendall et al.) e *test-time* adaptativo.
- **Radiômica** completa (textura/forma) unindo os biomarcadores à cabeça neural.